# Analisi esperimenti numerici

Un piccolo notebook per analizzare i dati degli esperimenti numerici, che salvo in un file `.csv`

## Imports, creazione del dataframe

In [1]:
import os 
import pandas as pd


In [2]:
CSV_FILENAME = "bench-v0.1.2alpha-5_05.csv"

In [3]:
df = pd.read_csv(CSV_FILENAME)

In [4]:
df.head()

,kind,n,eltype,approximant,algorithm,schur_time,alpha_time,eval_bound_time,eval_pade_time,squaring_time,...,delta,psi,cond_q,epsilon,rel_err_F,abs_err_1,nrm1_Ytrue,cond_expA_F,condA_1,condA_2
0,randn,16,Float64,scaling_and_squaring,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.220446e-16,6.287127e-16,2.642275e-12,3790.715547,NaN,1763.554439,782.38719
1,randn,16,Float64,diagonalcheap,transfree,0.000000,0.000214,0.348855,0.004563,0.000415,...,3.373615e-17,65.577306,17.070817,2.220446e-16,5.290437e-19,2.005686e-15,3790.715547,NaN,1763.554439,782.38719
2,randn,16,Float64,diagonalcheap,transfree,0.000000,0.000248,0.002303,0.010301,0.001924,...,3.373615e-17,65.577306,17.070817,2.220446e-16,5.290437e-19,2.005686e-15,3790.715547,NaN,1763.554439,782.38719
3,randn,16,Float64,diagonalcheap,transfree,0.000000,0.000300,0.002983,0.006556,0.000806,...,3.373615e-17,65.577306,17.070817,2.220446e-16,5.290437e-19,2.005686e-15,3790.715547,NaN,1763.554439,782.38719
4,randn,16,Float64,diagonalcheap,complexschur,0.015293,0.000431,1.643650,0.016344,0.002364,...,1.540473e-17,57.575904,13.654398,2.220446e-16,4.534437e-20,2.677012e-16,3790.715547,NaN,1763.554439,782.38719


## Descrizione colonne

In [5]:
df.columns

Index(['kind', 'n', 'eltype', 'approximant', 'algorithm', 'schur_time',
       'alpha_time', 'eval_bound_time', 'eval_pade_time', 'squaring_time',
       'total_time', 'm', 's', 'delta', 'psi', 'cond_q', 'epsilon',
       'rel_err_F', 'abs_err_1', 'nrm1_Ytrue', 'cond_expA_F', 'condA_1',
       'condA_2'],
      dtype='object')

Ecco una descrizione delle colonne:
- `kind`: un identificativo del "tipo" di matrice che ha prodotto i risultati della riga corrispondente
- `n`: dimensione della matrice 
- `eltype`: tipo degli elementi della matrice
- `approximant`: approssimante scelta per il calcolo dell'esponenziale di matrice 
- `algorithm`: algoritmo usato da `exp_mp`. Può essere Schur reale, Schur complesso o niente Schur
- `schur_time`: tempo impiegato per calcolare la forma di Schur
- `alpha_time`: tempo totale impiegato per valutare l'$\alpha_{\min}$ in `exp_mp`
- `eval_bound_time`: tempo totale impiegato per valutare il bound sull'errore in `exp_mp`. Si osservi che la valutazione del bound può formare nuove potenze di $A$
- `eval_pade_time`: tempo totale impiegato nel valutare l'approssimante 
- `squaring_time`: tempo speso nella fase di squaring
- `total_time`: tempo totale impiegato dall'algoritmo (viene misurato direttamente. Non è la somma dei tempi precedenti)
- `m`: grado dell'approssimante scelto dall'algoritmo `exp_mp`
- `s`: fattore di scaling scelto dall'algoritmo `exp_mp` ($A$ viene scalata di $2^{-s}$)
- `delta`: valore finale di $\delta$ (ossia upper bound sull'errore) prodotto in `exp_mp`
- `psi`: valore finale di $\psi$ (ossia, stima di $\exp(A)$ usata stimare per l'errore relativo durante l'esecuzione di `exp_mp`)
- `cond_q`: valore finale del numero di condizionamento (in norma 1) di $q_m(2^{-s}A)$
- `epsilon`: valore $\epsilon$ usato per la tolleranza sull'errore relativo. Può essere passato in input, il valore di default è la precisione di macchina della precisione usata all'interno dell'algoritmo
- `rel_err_F`: accuratezza relativa della soluzione calcolata, in norma di Frobenius
- `abs_err_1`: errore assoluto della soluzione calcolata, in norma 1
- `nrm1_Ytrue`: norma 1 della soluzione di riferimento $Y_{\textup{true}}$ (considerata esatta nella valutazione dell'errore relativo)
- `cond_expA_F`: $\kappa_{\exp}(A)$, numero di condizionamento dell'esponenziale di $A$, in norma di Frobenius (se $A$ è troppo grande, il valore è `NaN`. Infatti il calcolo è "esatto")
- `condA_1`: $\kappa_1(A)$, numero di condizionamento di $A$ in norma 1
- `condA_2`: $\kappa_2(A)$, numero di condizionamento di $A$ in norma 2

## Analisi

In [6]:
df.columns

Index(['kind', 'n', 'eltype', 'approximant', 'algorithm', 'schur_time',
       'alpha_time', 'eval_bound_time', 'eval_pade_time', 'squaring_time',
       'total_time', 'm', 's', 'delta', 'psi', 'cond_q', 'epsilon',
       'rel_err_F', 'abs_err_1', 'nrm1_Ytrue', 'cond_expA_F', 'condA_1',
       'condA_2'],
      dtype='object')

### $\delta$ è un upper bound?

La domanda più pressante che mi viene in mente è: ma alla fine, $\delta$ è un upper bound all'errore, come rivendicano gli autori dell'articolo? 

***OSS***: Consideriamo che per il calcolo di un termine di $\delta$ si usa una stima della norma 1 eh...

In [7]:
(df["abs_err_1"]/df["nrm1_Ytrue"]) <= df["delta"]

0     False
1      True
2      True
3      True
4      True
      ...  
71    False
72    False
73    False
74    False
75    False
Length: 76, dtype: bool

In [8]:
df[(df["abs_err_1"]/df["nrm1_Ytrue"]) > df["delta"]][["kind", "approximant", "algorithm", "abs_err_1", "nrm1_Ytrue", "delta", "epsilon"]]

,kind,approximant,algorithm,abs_err_1,nrm1_Ytrue,delta,epsilon
39,randn_big,diagonalcheap,transfree,1.820254e-73,3.084445e+03,2.447647e-82,1.727234e-77
40,randn_big,diagonalcheap,transfree,5.043836e-305,3.084445e+03,0.000000e+00,1.112537e-308
41,randn_big,diagonalcheap,transfree,2.906056e-252,3.084445e+03,5.439126e-279,1.331998e-256
42,randn_big,diagonalcheap,complexschur,2.490085e-72,3.084445e+03,2.141691e-82,1.727234e-77
43,randn_big,diagonalcheap,complexschur,3.530098e-303,3.084445e+03,0.000000e+00,1.112537e-308
44,randn_big,diagonalcheap,complexschur,5.974579e-251,3.084445e+03,5.036227e-280,1.331998e-256
45,randn_big,diagonalcheap,realschur,2.467977e-72,3.084445e+03,2.141691e-82,1.727234e-77
46,randn_big,diagonalcheap,realschur,3.557155e-303,3.084445e+03,0.000000e+00,1.112537e-308
47,randn_big,diagonalcheap,realschur,5.942185e-251,3.084445e+03,5.036227e-280,1.331998e-256
48,randn_big,taylor,transfree,3.777382e-74,3.084445e+03,3.294437e-83,1.727234e-77


In [9]:
df[df["kind"] == "randn_big"].shape

(38, 23)

In [10]:
pd.concat(
    [
        df["delta"],
        (df["abs_err_1"] / df["nrm1_Ytrue"]).rename("abs_err1_over_nrm1_Ytrue"),
    ],
    axis=1,
)

,delta,abs_err1_over_nrm1_Ytrue
0,NaN,6.970387e-16
1,3.373615e-17,5.291048e-19
2,3.373615e-17,5.291048e-19
3,3.373615e-17,5.291048e-19
4,1.540473e-17,7.062024e-20
...,...,...
71,0.000000e+00,3.825086e-306
72,3.110115e-277,5.529509e-254
73,4.941655e-83,3.468321e-75
74,0.000000e+00,3.811328e-306


In [11]:
df[["abs_err_1", "nrm1_Ytrue", "delta"]]

,abs_err_1,nrm1_Ytrue,delta
0,2.642275e-12,3.790716e+03,NaN
1,2.005686e-15,3.790716e+03,3.373615e-17
2,2.005686e-15,3.790716e+03,3.373615e-17
3,2.005686e-15,3.790716e+03,3.373615e-17
4,2.677012e-16,3.790716e+03,1.540473e-17
...,...,...,...
71,5.577454e-292,1.458125e+14,0.000000e+00
72,8.062715e-240,1.458125e+14,3.110115e-277
73,5.057245e-61,1.458125e+14,4.941655e-83
74,5.557393e-292,1.458125e+14,0.000000e+00
